# 第19课：RAG 检索流水线

本笔记本是课堂讲义。每个知识点包含：理论知识、案例代码、讲解、易错点与练习。综合练习 P1 使用课程根目录的教学运行时 [`agent_lab`](../agent_lab/README.md)。课后独立练习见 [chapter19_RAG检索流水线_课后练习.ipynb](chapter19_RAG检索流水线_课后练习.ipynb)。

**阶段定位**：阶段四 · 任务三 3.1 / 15分。15分

模型没有读过你们公司的手册。RAG 把文档切片、向量化（本课用词频向量教学）、入库，再把 Retriever 当成 Skill/Tool 给 Agent 按需调用。任务三 3.1 占 15 分。

## 学习目标

1. 按段落做 Chunking，生成 chunk_id。
2. 构建 Retriever，对续航类问题命中产品手册。
3. 把检索封装为标准 Tool 并成功调用。

## 学习知识点

| 分块 | 入库 | 调用 |
| --- | --- | --- |
| 按空行近似 80 字 | TF 向量 + 检索 | rag_search Tool |
| doc_id::idx | 命中 hits | ResultSchema.hits |

## 基础回顾与案例提问

1. **R.1** 把三份 md 全文塞进 system Prompt，手册变厚时会怎样？
2. **R.2** chunk_id 写成 0、1、2 全局自增，跨文档溯源时有何不便？
3. **R.3** Agent 不调用 Retriever 直接回答续航，算不算 3.1 通过？

教学检索不连云端 Embedding API。中文用汉字 unigram/bigram 分词。知识库副本在本课 `kb/`。

使用 Python 3；需要 `pydantic`。从本课文件夹启动内核。本课不要求 GPU，也不强制安装 `langgraph` / `openai`。未配置私有化端点时，`get_client()` 返回进程内 Fake。不要使用 pandas。综合练习不要抄 `experiment.py` 的整段答案，按题面逐步完成。


In [ ]:
# R.1–R.3: Write and verify your predictions here.


In [ ]:
import sys
from pathlib import Path

COURSE = Path.cwd().resolve()
if COURSE.name.startswith("第"):
    COURSE = COURSE.parent
if str(COURSE) not in sys.path:
    sys.path.insert(0, str(COURSE))
print("已加入路径:", COURSE)
print("请从本课文件夹启动内核。未配置 OPENAI_BASE_URL 时使用教学 Fake 端点，不要求 GPU。")


## 1. 流水线全景

### 理论知识

**文档 → 切片 → 索引 → 查询 → 命中文本。** Agent 只在需要时调用最后一步。

### 案例：加载语料


In [ ]:
from pathlib import Path
from agent_lab.rag import load_corpus, Retriever
KB = Path.cwd().parent / "agent_lab" / "knowledge"
chunks = load_corpus(KB)
print(len(chunks), [c.chunk_id for c in chunks])


### 讲解

应看到 manual / price / policy 的若干 chunk。条数过少说明没读到 md。

### 易错点与练习

1. **K1.1** 为何按空行切而不是按固定 10 个汉字切？课堂选择的代价是什么？
2. **K1.2** policy 与续航问题无关时仍入库，对吗？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 2. Chunking

### 理论知识

**每块要能独立引用。** chunk_id = doc_id + 序号。

### 案例：看第一块


In [ ]:
print(chunks[0].model_dump())


### 讲解

后文溯源必须带回这段 id，而不是“根据网上资料”。

### 易错点与练习

1. **K2.1** 同一文档第二段的 id 应如何递增？
2. **K2.2** 切得太碎或太长各有什么检索后果？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 3. 向量化（教学版）

### 理论知识

**本课用词频向量 + 余弦，外加 BM25 备着第 20 课。** 不是工业 Embedding，但接口一样：query in，hits out。

### 案例：检索续航


In [ ]:
retriever = Retriever(chunks)
hits = retriever.search("Widget-X 续航", k=3)
for h in hits:
    print(h.score, h.chunk_id, h.text[:40])


### 讲解

第一条应落在手册续航附近。若落到价格表，检查分词是否把中文拆没了。

### 易错点与练习

1. **K3.1** k=3 的吞吐含义是什么？
2. **K3.2** score 跨查询能直接比大小吗？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 4. 封装为 Tool

### 理论知识

**Retriever 对 Agent 来说只是又一枚 Tool。** Args=query，Result=hits。

### 案例：挂载


In [ ]:
from pydantic import BaseModel, Field
from agent_lab.tools import Tool

class SearchArgs(BaseModel):
    query: str = Field(min_length=2)

class SearchResult(BaseModel):
    ok: bool = True
    error: str = ""
    hits: list = Field(default_factory=list)

def _search(args: SearchArgs):
    found = retriever.search(args.query, k=3)
    return {"ok": True, "error": "", "hits": [h.model_dump() for h in found]}

rag_tool = Tool("rag_search", "从产品知识库检索", SearchArgs, SearchResult, _search)
result = rag_tool.run(query="Widget-X 续航")
print(result.ok, result.hits[0]["chunk_id"])


### 讲解

15 分要求：独立服务可用（本课即 Retriever 对象）且成功被这种调用碰到。

### 易错点与练习

1. **K4.1** query 一个字为何要失败？
2. **K4.2** hits 放进 ResultSchema 而不是 print，是为了谁？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 5. 命中率抽检

### 理论知识

**续航问题必须能在 hits 文本里看到续航或 48。**

### 案例：断言思路


In [ ]:
assert result.ok and result.hits
assert any("续航" in h["text"] or "48" in h["text"] for h in result.hits)
print("抽检通过")


### 讲解

课堂知识库很小，抽检不是刷榜。换自己的文档后要重新选探针问题。

### 易错点与练习

1. **K5.1** 只用英文 Widget-X 不写续航，还保证命中手册吗？
2. **K5.2** 命中价格表算不算本探针失败？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 6. 给 Agent 按需调用

### 理论知识

**不是每轮都检索。** 问候语不必查库；事实型产品问题才调用 rag_search。对照第 14 课 need_tool。

### 案例：意图粗分


In [ ]:
def need_rag(text):
    keys = ["续航", "价格", "质保", "Widget"]
    return any(k in text for k in keys)
print(need_rag("你好"), need_rag("Widget-X 续航多久"))


### 讲解

真实系统会用更稳的路由。本课只要你意识到 Retriever 是节点上的工具。

### 易错点与练习

1. **K6.1** 无条件每轮检索的坏处？
2. **K6.2** 检索结果要不要再进 Pydantic SourcedAnswer？第 20 课做。

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 7. 本课 kb/ 副本

### 理论知识

**课后独立读取本课 kb/，避免讲义变量残留。** 内容与 agent_lab/knowledge 一致。

### 案例：列出本课 kb


In [ ]:
from pathlib import Path
print(sorted(p.name for p in Path("kb").glob("*.md")))


### 讲解

三份：manual.md / price.md / policy.md。不要上网另下语料。

### 易错点与练习

1. **K7.1** 验收脚本读的是哪套路径？
2. **K7.2** 你改了 kb 但没改 agent_lab/knowledge，experiment.py 会变吗？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 8. 15 分评分对照

### 理论知识

**流水线可用 + Agent 调用 + 探针命中。**

### 案例：回顾三项


In [ ]:
print("chunks", len(chunks))
print("tool", rag_tool.name)
print("hit0", result.hits[0]["chunk_id"])


### 讲解

调优、混合检索、幻觉兜底留给第 20 课，本课不要提前把 fallback 当主线。

### 易错点与练习

1. **K8.1** 没有 Tool 封装、只在笔记本 print hits，会缺哪条评分？
2. **K8.2** 吞吐在本课如何体现（k 路检索）？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 综合练习：检索服务挂到 Tool

按 P1.1 → P1.2 → P1.3 顺序完成。每步都要能独立看出你做了什么。


### P1.1　分块

load_corpus，打印分块数与前三个 chunk_id。


In [ ]:
# P1.1: chunking.


### P1.2　检索

查询 Widget-X 续航，打印 top hits。


In [ ]:
# P1.2: search.


### P1.3　Tool 调用

封装 rag_search 并 run，确认命中续航或 48。


In [ ]:
# P1.3: retriever as tool.


课后请打开 [chapter19_RAG检索流水线_课后练习.ipynb](chapter19_RAG检索流水线_课后练习.ipynb)，读取本课 [kb/](kb)。P1 自己建索引，P2 封装 Tool，P3 选做另一探针。
